# Build Inference Artifact for Churn Scoring API

## Purpose

This notebook creates a **production-ready inference artifact** from previously trained models and feature engineering steps.

The goal is to transform our notebook-based experimentation workflow into a **clean, reusable, and low-latency scoring pipeline** that can be used by the API.

---

## Background

Up to this point:
- Data preparation and feature engineering were implemented across multiple notebooks
- Models were trained and saved as `.pkl` files
- Feature engineering logic was partially externalized into JSON artifacts
- Preprocessing components (e.g. scaler, categorical setup) are embedded in the saved model bundle

However, these components are **not yet unified into a single inference-ready object**.

---

## Objective

The objective of this notebook is to:

1. **Load the trained model bundle**
   - Extract model, scaler, and feature preparation inputs

2. **Load feature engineering artifacts**
   - JSON-based transformation steps (releveling, aggregations, interactions, etc.)

3. **Reconstruct the full inference pipeline**
   - Raw input → feature engineering → encoding → scaling → model scoring

4. **Standardize all components into a single artifact**
   - Ensure consistent structure and naming
   - Define required input schema
   - Store metadata for traceability

5. **Validate the pipeline**
   - Run test inference on sample input
   - Verify feature alignment and prediction output

6. **Save the final inference artifact**
   - This artifact will be used directly by the API for scoring

---

## Target Output

The final artifact should encapsulate everything required for inference:

- Trained model
- Fitted scaler (if applicable)
- Feature engineering transformation rules
- Feature preparation parameters (categorical columns, reference columns, etc.)
- Expected input schema
- Decision threshold
- Metadata (model name, version, metrics)

This ensures that the API does not need to reconstruct any logic at runtime.

---

## Why This Matters

This step is critical for transitioning from experimentation to a **deployable ML system**:

- Removes dependency on notebooks during inference
- Ensures reproducibility and consistency
- Reduces latency by pre-packaging transformations
- Enables model versioning and auditability
- Provides a clear contract between model and API

---

## Scope

This notebook focuses on a **single churn prediction model (v1)**.

Future model improvements or retraining cycles will result in new versions (e.g. v2), each with its own inference artifact.

We are not building a generic ML platform — this is a **focused, production-style implementation for a specific business use case**.

---

## Outcome

After completing this notebook:
- We will have a **single, standardized inference artifact**
- The API will be able to load this artifact and serve predictions reliably
- The project will move from "modeling" to **"model serving and lifecycle management"**

In [ ]:
import os
import json
import joblib
from pathlib import Path

# CONFIG

MODEL_NAME = "churn_hist_gradient_boosting"
MODEL_VERSION = "v1"
THRESHOLD = 0.31

SOURCE_MODEL_PATH = Path("../artifacts/source/model_bundles/hist_boosted_trees_final.pkl")
FE_ARTIFACTS_DIR = Path("../artifacts/source/feature_engineering")

OUTPUT_DIR = Path(f"../artifacts/serving/{MODEL_NAME}/{MODEL_VERSION}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_ARTIFACT_PATH = OUTPUT_DIR / "inference_artifact.joblib"
OUTPUT_METADATA_PATH = OUTPUT_DIR / "metadata.json"


# 1. LOAD MODEL BUNDLE

print("Loading model bundle...")
model_bundle = joblib.load(SOURCE_MODEL_PATH)

# Inspect structure (optional)
print("Model bundle keys:", list(model_bundle.keys()) if isinstance(model_bundle, dict) else type(model_bundle))


# 2. LOAD FEATURE ARTIFACTS

print("Loading feature engineering artifacts...")

fe_artifacts = []

# ensure ordering (important!)
artifact_files = sorted(FE_ARTIFACTS_DIR.glob("*.json"))

for file_path in artifact_files:
    with open(file_path, "r") as f:
        artifact = json.load(f)
        fe_artifacts.append(artifact)

print(f"Loaded {len(fe_artifacts)} feature artifacts")


# 3. EXTRACT COMPONENTS FROM MODEL BUNDLE
if isinstance(model_bundle, dict):
    model = model_bundle.get("model") or model_bundle.get("clf")
    scaler = model_bundle.get("scaler")
    categorical_cols = model_bundle.get("categorical_cols")
    numerical_cols = model_bundle.get("numerical_cols")
    categorical_orders = model_bundle.get("categorical_orders")
    reference_columns = model_bundle.get("reference_columns")
else:
    raise ValueError("Model bundle structure not recognized")

print("Model extracted:", type(model))
print("Scaler present:", scaler is not None)


# 4. BUILD SERVING ARTIFACT

serving_artifact = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_family": type(model).__name__,
    "model": model,
    "scaler": scaler,
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "categorical_orders": categorical_orders,
    "reference_columns": reference_columns,
    "feature_engineering_artifacts": fe_artifacts,
    "threshold": THRESHOLD,
}


# 5. SAVE ARTIFACT

print("Saving inference artifact...")
joblib.dump(serving_artifact, OUTPUT_ARTIFACT_PATH)

print(f"Saved artifact to: {OUTPUT_ARTIFACT_PATH}")


# 6. SAVE METADATA

metadata = {
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "model_family": type(model).__name__,
    "threshold": THRESHOLD,
    "source_model_path": str(SOURCE_MODEL_PATH),
    "feature_artifacts": [str(p) for p in artifact_files],
}

with open(OUTPUT_METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata to: {OUTPUT_METADATA_PATH}")


# DONE

print("Inference artifact build complete.")

Loading model bundle...
Model bundle keys: ['model', 'categorical_cols', 'numerical_cols', 'categorical_orders', 'target_col', 'reference_columns']
Loading feature engineering artifacts...
Loaded 6 feature artifacts
Model extracted: <class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>
Scaler present: False
Saving inference artifact...
Saved artifact to: artifacts\serving\churn_hist_gradient_boosting\v1\inference_artifact.joblib
Saved metadata to: artifacts\serving\churn_hist_gradient_boosting\v1\metadata.json
Inference artifact build complete.
